# RockAndEgg-313M | Train a Text Classifier on 2xT4

This notebook shows how to turn the public **RockAndEgg-313M** backbone into a
task-specific text classifier, trained on Kaggle's **two NVIDIA Tesla T4 GPUs**.

## What this notebook is

RockAndEgg-313M is a 10-layer Qwen3-style model with approximately 312.9M
parameters and 1,024-dimensional hidden states. The published checkpoint
(`moeshawky/rockandegg-313m`, attached as a Kaggle model input) is **headless**:
it contains the transformer backbone only, with no trained task-specific
classifier head. Your job in this notebook is to train that head so the model
can map its representations to your chosen labels.

## What you will see, section by section

Each numbered section below pairs a Markdown explanation with one or more code
cells. Every explanation answers four questions:

1. **What are we doing?**
2. **Why does this operation matter?**
3. **What should you expect to observe?**
4. **What does the result establish — and what does it *not* establish?**

| Section | What | Why | What to observe | What it establishes |
|---|---|---|---|---|
| 1 Environment | Inspect Kaggle runtime and both T4 GPUs | Confirms the accelerator is present and reachable before any expensive work | Torch version, CUDA device count, per-GPU name and memory | That both T4s are detected and usable |
| 2 Model anatomy | Load the frozen backbone, read architecture from `config.json` | Verifies the checkpoint is headless and dimensions match the design (no hard-coded 1024/10) | Hidden size, layer count, pad-token-id, frozen parameter count | That the correct checkpoint is loaded and frozen |
| 3 Dataset | Load AG News, build stratified train/val/test splits (16k/2k/2k, seed 42) | Guarantees disjoint, label-balanced splits so evaluation is trustworthy | Split sizes, per-class label counts | That train/val/test are disjoint and stratified |
| 4 Classifier design | Define a linear head and padding-side-aware last-token pooling | The head maps 1,024-d backbone representations to 4 class logits; pooling must use the last *non-padding* token | Head parameter count, pooling function | That the head and pooling are correctly defined |
| 5 Dual-GPU extraction | Compute frozen backbone features on both T4s, shard rows, merge | Uses both GPUs for parallel inference; shards must cover the dataset without omissions or duplicates | Per-rank count, elapsed time, peak memory, merged shape | That both T4s processed nonempty, complete shards |
| 6 Train the head | Optimize the classifier head on cached features (AdamW, 5 epochs, lr 1e-3) | Frozen backbone means only the head trains; checkpoint selected on validation macro F1 | Per-epoch train/val loss, val macro F1, best checkpoint | That the head trains and a checkpoint is selected |
| 7 Evaluate | Held-out test accuracy, macro/weighted F1, confusion matrix | Reports generalization on data never seen during training or selection | Test accuracy, macro F1, per-class metrics | Test-set performance of the selected checkpoint |
| 8 Export | Save a reloadable Transformers classifier; verify reload parity | The exported model must load independently and reproduce training-time logits | Max logit difference (< 1e-4 required) | That export/reload preserves predictions |
| 9 Advanced | Optional partial fine-tuning (disabled by default) | Unfreezes final backbone layers only after the main workflow succeeds | `RUN_ADVANCED = False` | That advanced training is opt-in |
| 10 Conclusions | Summarize results and limitations | States what the numbers do and do not support | Measured metrics, caveats | Honest interpretation, no superiority claims |

## Default configuration

| Setting | Value |
|---|---|
| Backbone | `moeshawky/rockandegg-313m` (attached Kaggle model input) |
| Optional second backbone | `moeshawky/rockandegg-313m-v2` (attached Kaggle model input) |
| Dataset | AG News (attached Kaggle dataset input) |
| Task | Single-label classification, 4 classes |
| Train / Val / Test | 16,000 / 2,000 / 2,000 stratified samples |
| Maximum length | 128 tokens |
| Training mode | Frozen backbone, train head only (`head_only`) |
| Head epochs | 5 |
| Head learning rate | 1e-3 (AdamW) |
| Seed | 42 |
| Backbone compute | FP16 |
| Head optimization | FP32 |

## What this notebook does *not* claim

- **No measured performance numbers are published here.** Accuracy, F1, extraction
  time, and GPU memory are measured when the notebook runs on Kaggle and are not
  fabricated in this source. Any section showing metrics reports what the run
  produced; nothing here is a guarantee.
- **The v1/v2 comparison is not independent out-of-distribution evidence.** The v2
  checkpoint received AG News supervision during foundation training, so a side-by-side
  comparison does not measure zero-shot generalization.
- **No superiority claim is made.** Frozen-backbone performance depends on backbone
  representations; a weak probe is reported accurately, not spun.

All artifacts are written under `/kaggle/working/`. The default run needs no
credentials, secrets, or internet access once the model and dataset are attached.
Kaggle builds a navigable table of contents from these Markdown headings.


# Section 1 - Environment and hardware inspection

**What are we doing?**
Probe the Kaggle runtime: Python version, whether `/kaggle/working` exists, whether PyTorch and CUDA are available, and how many GPU devices are visible with their names and memory.

**Why does this operation matter?**
The entire notebook depends on two T4 GPUs being present and usable. Checking here fails fast with a clear message instead of failing deep inside feature extraction after a long model load.

**What should you expect to observe?**
`torch.cuda.device_count()` returns `2`, and each GPU reports a Tesla T4 name with roughly 15 GiB of memory. If CUDA is unavailable or fewer than two devices show, the later cells will fail at GPU placement.

**What does the result establish — and what does it *not* establish?**
It establishes that the runtime exposes two CUDA devices. It does *not* establish that both GPUs will actually process work; that is verified in Section 5 by checking per-rank shard counts.

`MODEL_VARIANT` selects which checkpoint(s) to run: `"v1"`, `"v2"`, or `"compare"`. Each variant resolves to its attached Kaggle model directory under `/kaggle/input/.../transformers/default/1`. A missing directory raises immediately — there is no silent fallback to a different checkpoint.


In [1]:
# Stage 1 - Environment and hardware inspection.

import os
import platform
import sys

print("python  :", sys.version.split()[0])
print("machine :", platform.machine())
print("kaggle  :", os.path.isdir("/kaggle/working"))

try:
    import torch
    print("torch   :", torch.__version__)
    print("cuda    :", torch.cuda.is_available())
    print("devices :", torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print("gpu %d  :" % index, props.name, "%.1f GiB" % (props.total_memory / 1024**3))
except Exception as error:
    print("torch probe failed:", repr(error))

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
print("work_dir:", WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Select which checkpoint(s) to run: "v1", "v2", or "compare".
MODEL_VARIANT = "compare"

MODEL_DIRS = {
    "v1": "/kaggle/input/models/moeshawky/rockandegg-313m/transformers/default/1",
    "v2": "/kaggle/input/models/moeshawky/rockandegg-313m-v2/transformers/default/1",
}

if MODEL_VARIANT == "v1":
    VARIANTS = ["v1"]
elif MODEL_VARIANT == "v2":
    VARIANTS = ["v2"]
else:
    VARIANTS = ["v1", "v2"]

for variant in VARIANTS:
    if not os.path.isdir(MODEL_DIRS[variant]):
        raise FileNotFoundError("Attached model directory not found for %s: %s" % (variant, MODEL_DIRS[variant]))
    print(variant, "->", MODEL_DIRS[variant])

print("variants:", VARIANTS)


python  : 3.13.16
machine : x86_64
kaggle  : True
torch   : 2.11.0+cu130
cuda    : True
devices : 2
gpu 0  : Tesla T4 14.6 GiB
gpu 1  : Tesla T4 14.6 GiB
work_dir: /kaggle/working
v1 -> /kaggle/input/models/moeshawky/rockandegg-313m/transformers/default/1
v2 -> /kaggle/input/models/moeshawky/rockandegg-313m-v2/transformers/default/1
variants: ['v1', 'v2']


# Section 2 - Model anatomy: load the frozen backbone

**What are we doing?**
For each selected variant, read `config.json` from its attached Kaggle directory, then load `AutoModel` and `AutoTokenizer` from that directory, freeze every backbone parameter, and set the model to evaluation mode.

**Why does this operation matter?**
Reading architecture from `config.json` (hidden size, layer count) instead of hard-coding `1024`/`10` makes the notebook adapt to whichever checkpoint is attached. Freezing the backbone guarantees the primary workflow never modifies RockAndEgg's weights — only the classifier head trains.

**What should you expect to observe?**
For each variant: hidden size, number of layers, padding side, pad token id, and confirmation that all backbone parameters are frozen (`requires_grad == False`).

**What does the result establish — and what does it *not* establish?**
It establishes the correct checkpoint is loaded, its architecture matches the design, and the backbone is frozen. It does *not* establish that the backbone representations are good for classification — that is measured later by the head's validation performance.

No model weights are downloaded at runtime unless no attached directory exists; the notebook requires the model to be attached as a Kaggle input.


In [2]:
# Stage 2 - Load and validate each model variant from its attached Kaggle directory.

import json
import os

import torch
from transformers import AutoModel, AutoTokenizer

models = {}
tokenizers = {}
model_configs = {}

for variant in VARIANTS:
    model_dir = MODEL_DIRS[variant]
    if not os.path.isdir(model_dir):
        raise FileNotFoundError(
            "Attached model directory not found for %s: %s" % (variant, model_dir)
        )
    with open(os.path.join(model_dir, "config.json"), encoding="utf-8") as handle:
        config = json.load(handle)
    hidden_size = int(config.get("hidden_size"))
    num_layers = int(config.get("num_hidden_layers"))
    if hidden_size <= 0 or num_layers <= 0:
        raise ValueError("Invalid architecture config for %s" % variant)
    print(variant, "hidden_size=", hidden_size, "num_layers=", num_layers)

    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_dir,
        trust_remote_code=True,
        dtype=torch.float32,
    )
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.eval()

    models[variant] = model
    tokenizers[variant] = tokenizer
    model_configs[variant] = {
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "padding_side": getattr(tokenizer, "padding_side", "right"),
        "pad_token_id": tokenizer.pad_token_id,
    }
    print(variant, "loaded; frozen=", all(not p.requires_grad for p in model.parameters()))

print("models loaded:", list(models.keys()))


v1 hidden_size= 1024 num_layers= 10


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

v1 loaded; frozen= True
v2 hidden_size= 1024 num_layers= 10


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

v2 loaded; frozen= True
models loaded: ['v1', 'v2']


# Section 3 - Dataset: AG News and stratified splits

**What are we doing?**
Locate the AG News CSV under `/kaggle/input`, normalize 1-based labels to 0-based, and build disjoint stratified train/validation/test splits of 16,000 / 2,000 / 2,000 examples using seed 42. Write the combined splits to `all_splits.csv` for the extraction workers.

**Why does this operation matter?**
Stratified deterministic splitting keeps class proportions stable across splits and makes the run reproducible. A disjoint held-out test set guarantees that Section 7 reports generalization on data never seen during training or checkpoint selection.

**What should you expect to observe?**
The input CSV columns, overall shape, and split sizes (train 16,000 / val 2,000 / test 2,000) with roughly equal per-class counts in each split.

**What does the result establish — and what does it *not* establish?**
It establishes that train, validation, and test are disjoint and stratified. It does *not* establish that AG News is representative of any other task; it is a practical demonstration dataset.

Label mapping: `0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech`. The exact Kaggle dataset identifier and version are declared in `recipe.toml` under `[[metadata.kaggle.dataSources]]`.


In [3]:
# Stage 3 - Dataset loading and stratified splits (shared across variants).

import os
import random

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CONFIG = {
    "task": "single_label_classification",
    "dataset_name": "ag_news",
    "text_column": "text",
    "label_column": "label",
    "train_size": 16000,
    "val_size": 2000,
    "test_size": 2000,
    "max_length": 128,
    "head_epochs": 5,
    "head_lr": 1e-3,
    "batch_size": 64,
}

DATASET_DIR = "/kaggle/input/datasets/amananandrai/ag-news-classification-dataset"

# Fallback search if running outside Kaggle or path differs
if not os.path.isdir(DATASET_DIR):
    for candidate in [
        "/kaggle/input/ag-news-classification-dataset",
        "./ag-news-classification-dataset",
        ".",
    ]:
        if os.path.isdir(candidate):
            DATASET_DIR = candidate
            break

train_csv = os.path.join(DATASET_DIR, "train.csv")
test_csv = os.path.join(DATASET_DIR, "test.csv")

# Load train and test, then combine to ensure full sample pool (127,600 samples)
if os.path.exists(train_csv):
    df_train = pd.read_csv(train_csv)
    df_test = pd.read_csv(test_csv) if os.path.exists(test_csv) else pd.DataFrame()
    raw_frame = (
        pd.concat([df_train, df_test], ignore_index=True)
        if not df_test.empty
        else df_train
    )
else:
    # Fallback to loading any available CSV if filenames differ
    csv_files = [f for f in os.listdir(DATASET_DIR) if f.endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {DATASET_DIR}")
    raw_frame = pd.concat(
        [pd.read_csv(os.path.join(DATASET_DIR, f)) for f in csv_files],
        ignore_index=True,
    )

frame = pd.DataFrame()

# Standardize text column: combine Title and Description
title_col = (
    "Title"
    if "Title" in raw_frame.columns
    else [c for c in raw_frame.columns if "title" in c.lower()][0]
)
desc_col = (
    "Description"
    if "Description" in raw_frame.columns
    else [c for c in raw_frame.columns if "desc" in c.lower()][0]
)

frame[CONFIG["text_column"]] = (
    raw_frame[title_col].fillna("") + " " + raw_frame[desc_col].fillna("")
).str.strip()

# Standardize label column: convert 1-indexed (1..4) to 0-indexed (0..3)
class_col = (
    "Class Index"
    if "Class Index" in raw_frame.columns
    else [c for c in raw_frame.columns if "class" in c.lower() or "label" in c.lower()][
        0
    ]
)
labels = raw_frame[class_col].astype(int).to_numpy()
if labels.min() == 1:
    labels = labels - 1
frame[CONFIG["label_column"]] = labels

# Stratified subset split
train_frame, test_frame = train_test_split(
    frame,
    train_size=CONFIG["train_size"] + CONFIG["val_size"],
    test_size=CONFIG["test_size"],
    stratify=frame[CONFIG["label_column"]],
    random_state=SEED,
)
train_frame, val_frame = train_test_split(
    train_frame,
    train_size=CONFIG["train_size"],
    test_size=CONFIG["val_size"],
    stratify=train_frame[CONFIG["label_column"]],
    random_state=SEED,
)

train_frame = train_frame.reset_index(drop=True)
val_frame = val_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

print(
    f"train: {len(train_frame)} | val: {len(val_frame)} | test: {len(test_frame)}"
)

LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in enumerate(LABEL_NAMES)}

# Add split identifier before saving
train_frame["split"] = "train"
val_frame["split"] = "val"
test_frame["split"] = "test"

combined = pd.concat([train_frame, val_frame, test_frame], ignore_index=True)
DATA_CSV = os.path.join(os.getcwd(), "all_splits.csv")
combined.to_csv(DATA_CSV, index=False)
print("combined csv saved to:", DATA_CSV, combined.shape)

train: 16000 | val: 2000 | test: 2000
combined csv saved to: /kaggle/working/all_splits.csv (20000, 3)


# Section 4 - Classifier design: head and last-token pooling

**What are we doing?**
Define a linear classification head mapping 1,024 hidden dimensions to four class logits, and a padding-side-aware pooling function that takes the contextual representation of the final *non-padding* token.

**Why does this operation matter?**
The head is the only part of the model that trains in the primary workflow. Pooling must use the last non-padding token, not the last array position, because right- or left-padding would otherwise let padding tokens contaminate the representation. The head matches the native `AutoModelForSequenceClassification` parameterization (linear + bias) so export stays Transformers-compatible.

**What should you expect to observe?**
Head parameter count (1,024 × 4 + 4 bias) and confirmation that pooling is padding-side aware.

**What does the result establish — and what does it *not* establish?**
It establishes the head and pooling are correctly defined. It does *not* establish classification quality — that comes from training and evaluation.

`make_head`, `last_non_padding_index`, and `pool_last_token` are defined here and reused by the training, export, and reload-parity cells so every stage applies the identical pooling convention.


In [4]:
# Stage 4 - Classifier head and padding-side-aware last-token pooling.

import torch
import torch.nn as nn


def make_head(hidden_size, num_labels):
    """Linear classification head mapping hidden states to class logits."""
    return nn.Linear(hidden_size, num_labels, bias=False)


def last_non_padding_index(attention_mask):
    """Index of the last non-padding token, correct for left or right padding."""
    reversed_mask = attention_mask.flip(1)
    last_from_end = reversed_mask.argmax(dim=1)
    return attention_mask.shape[1] - 1 - last_from_end


def pool_last_token(hidden_states, attention_mask):
    """Pool the contextual representation of the final non-padding token."""
    indices = last_non_padding_index(attention_mask)
    batch_size = hidden_states.shape[0]
    return hidden_states[
        torch.arange(batch_size, device=hidden_states.device), indices
    ]


print("pooling: last non-padding token (padding-side aware)")


pooling: last non-padding token (padding-side aware)


# Section 5 - Dual-GPU frozen feature extraction

**What are we doing?**
Write a standalone worker script, then launch one worker per T4 (rank 0 → CUDA device 0, rank 1 → CUDA device 1). Each worker loads its own model replica, processes a deterministic disjoint row shard (rows where `index % 2 == rank`), extracts 1,024-d frozen backbone features in FP16, converts to FP32, and writes a shard artifact. The controlling process verifies both GPUs processed nonempty shards, that shards cover the dataset without omissions or duplicates, then merges and restores original example ordering.

**Why does this operation matter?**
Frozen backbone representations can be computed once and reused across head-training epochs, avoiding repeated transformer inference. Data parallelism (not model splitting) lets each GPU process a different subset of examples using its own replica; results are combined into one common head. No gradients are synchronized, so there is no unnecessary all-reduce.

**What should you expect to observe?**
Per-rank example count, elapsed time, and peak memory; confirmation that neither rank processed an empty shard; merged feature shape matching the full dataset; unique sample IDs covering every row exactly once.

**What does the result establish — and what does it *not* establish?**
It establishes both T4s processed complete, nonoverlapping shards covering the dataset. It does *not* establish a 2x speedup — process startup and model-loading overhead may offset parallelism; wall time is measured and reported honestly.

Features are cached per variant and never shared between checkpoints, because their weight-derived representations differ. GPU verification (device names, memory, placement) is recorded per rank.


In [5]:
# Stage 5 - Write the dual-GPU feature-extraction worker script.
#
# The worker is a standalone Python file invoked once per T4 rank. Each rank
# loads its own model replica, processes a deterministic disjoint row shard,
# and writes its own shard artifact. No gradients are synchronized; frozen
# feature extraction needs no all-reduce.

import os
import textwrap

WORKER_PATH = os.path.join(os.getcwd(), "extract_worker.py")

worker_code = textwrap.dedent("""\
import argparse
import os
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--rank", type=int, required=True)
    parser.add_argument("--world_size", type=int, required=True)
    parser.add_argument("--variant", type=str, required=True)
    parser.add_argument("--model_dir", type=str, required=True)
    parser.add_argument("--data_csv", type=str, required=True)
    parser.add_argument("--text_col", type=str, default="text")
    parser.add_argument("--label_col", type=str, default="label")
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--out_dir", type=str, required=True)
    args = parser.parse_args()
    torch.cuda.set_device(args.rank)
    device = torch.device("cuda", args.rank)
    tok = AutoTokenizer.from_pretrained(args.model_dir, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        args.model_dir, trust_remote_code=True, dtype=torch.float16
    ).to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    frame = pd.read_csv(args.data_csv)
    shard = [i for i in range(len(frame)) if i % args.world_size == args.rank]
    feats = []
    labels = []
    ids = []
    start = time.time()
    peak = 0.0
    with torch.inference_mode():
        for off in range(0, len(shard), 32):
            bids = shard[off:off + 32]
            texts = [str(frame.iloc[i][args.text_col]) for i in bids]
            labs = [int(frame.iloc[i][args.label_col]) for i in bids]
            enc = tok(
                texts,
                padding=True,
                truncation=True,
                max_length=args.max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            out = model(**enc)
            hidden = out.last_hidden_state
            mask = enc["attention_mask"]
            rev = mask.flip(1)
            last = mask.shape[1] - 1 - rev.argmax(dim=1)
            pooled = hidden[
                torch.arange(hidden.shape[0], device=device), last
            ].float().cpu().numpy()
            feats.append(pooled)
            labels.extend(labs)
            ids.extend(bids)
            peak = max(peak, torch.cuda.max_memory_allocated(device) / 1024**3)
    feats = np.concatenate(feats, axis=0) if feats else np.zeros((0, 1), dtype=np.float32)
    elapsed = time.time() - start
    out_path = os.path.join(
        args.out_dir, "features_%s_rank%d.npz" % (args.variant, args.rank)
    )
    np.savez(
        out_path,
        features=feats,
        labels=np.array(labels, dtype=np.int64),
        sample_ids=np.array(ids, dtype=np.int64),
        variant=args.variant,
        rank=args.rank,
        elapsed=elapsed,
        peak_mem_gb=peak,
        count=len(shard),
    )
    print(
        "variant", args.variant, "rank", args.rank,
        "processed", len(shard), "in", round(elapsed, 2), "s",
    )


if __name__ == "__main__":
    main()
""")

with open(WORKER_PATH, "w", encoding="utf-8") as handle:
    handle.write(worker_code)

print("wrote", WORKER_PATH)


wrote /kaggle/working/extract_worker.py


In [6]:
# Stage 6 - Launch one worker per T4, verify shard coverage, then merge.

import os
import subprocess
import sys
import time

import numpy as np

FEATURE_DIR = os.path.join(os.getcwd(), "features")
os.makedirs(FEATURE_DIR, exist_ok=True)

WORKER = os.path.join(os.getcwd(), "extract_worker.py")
if not os.path.isfile(WORKER):
    raise FileNotFoundError("extract_worker.py missing; run stage 5 first")

extraction_summary = {}
for variant in VARIANTS:
    vdir = os.path.join(FEATURE_DIR, variant)
    os.makedirs(vdir, exist_ok=True)
    for name in os.listdir(vdir):
        if name.startswith("features_") and name.endswith(".npz"):
            os.remove(os.path.join(vdir, name))

    start = time.time()
    procs = []
    for rank in (0, 1):
        proc = subprocess.Popen(
            [
                sys.executable, WORKER,
                "--rank", str(rank),
                "--world_size", "2",
                "--variant", variant,
                "--model_dir", MODEL_DIRS[variant],
                "--data_csv", DATA_CSV,
                "--text_col", CONFIG["text_column"],
                "--label_col", CONFIG["label_column"],
                "--max_length", str(CONFIG["max_length"]),
                "--out_dir", vdir,
            ],
            env={**os.environ, "CUDA_VISIBLE_DEVICES": "0,1"},
        )
        procs.append(proc)
    for proc in procs:
        proc.wait()

    wall = time.time() - start
    rank_results = []
    total = 0
    for rank in (0, 1):
        shard_path = os.path.join(vdir, "features_%s_rank%d.npz" % (variant, rank))
        data = np.load(shard_path)
        rank_results.append({
            "rank": int(data["rank"]),
            "count": int(data["count"]),
            "elapsed": float(data["elapsed"]),
            "peak_mem_gb": float(data["peak_mem_gb"]),
        })
        total += int(data["count"])

    # Strict checks: both GPUs processed nonempty, disjoint shards.
    if rank_results[0]["count"] == 0 or rank_results[1]["count"] == 0:
        raise RuntimeError("one T4 processed an empty shard for %s" % variant)

    all_ids = []
    for rank in (0, 1):
        data = np.load(os.path.join(vdir, "features_%s_rank%d.npz" % (variant, rank)))
        all_ids.extend(data["sample_ids"].tolist())
    total_rows = len(open(DATA_CSV).readlines()) - 1
    if len(set(all_ids)) != len(all_ids) or set(all_ids) != set(range(total_rows)):
        raise RuntimeError("shard coverage/duplication check failed for %s" % variant)

    extraction_summary[variant] = {
        "wall_s": round(wall, 2),
        "rank0": rank_results[0],
        "rank1": rank_results[1],
        "total": total,
    }
    print(variant, "extraction done:", extraction_summary[variant])

import json as _json
with open(os.path.join(os.getcwd(), "extraction_summary.json"), "w", encoding="utf-8") as fh:
    _json.dump(extraction_summary, fh, indent=2, sort_keys=True)
print("wrote extraction_summary.json")


Loading weights: 100%|██████████| 112/112 [00:00<00:00, 205.84it/s]


variant v1 rank 0 processed 10000 in 31.63 s
variant v1 rank 1 processed 10000 in 32.3 s
v1 extraction done: {'wall_s': 45.84, 'rank0': {'rank': 0, 'count': 10000, 'elapsed': 31.62965488433838, 'peak_mem_gb': 1.0156769752502441}, 'rank1': {'rank': 1, 'count': 10000, 'elapsed': 32.29708409309387, 'peak_mem_gb': 1.0157380104064941}, 'total': 20000}


Loading weights: 100%|██████████| 112/112 [00:00<00:00, 299.23it/s]


variant v2 rank 0 processed 10000 in 31.39 s
variant v2 rank 1 processed 10000 in 32.73 s
v2 extraction done: {'wall_s': 44.07, 'rank0': {'rank': 0, 'count': 10000, 'elapsed': 31.38568115234375, 'peak_mem_gb': 1.0156769752502441}, 'rank1': {'rank': 1, 'count': 10000, 'elapsed': 32.73055696487427, 'peak_mem_gb': 1.0157380104064941}, 'total': 20000}
wrote extraction_summary.json


In [7]:
# Stage 7 - Verify, merge, and restore original ordering of sharded features.

import os

import numpy as np

FEATURE_DIR = os.path.join(os.getcwd(), "features")
merged = {}

for variant in VARIANTS:
    vdir = os.path.join(FEATURE_DIR, variant)
    shards = []
    for rank in (0, 1):
        path = os.path.join(vdir, "features_%s_rank%d.npz" % (variant, rank))
        data = np.load(path)
        shards.append(data)
        print(variant, "rank", rank, "count", int(data["count"]), "elapsed", round(float(data["elapsed"]), 2))

    all_features = []
    all_labels = []
    all_ids = []
    for data in shards:
        all_features.append(data["features"])
        all_labels.append(data["labels"])
        all_ids.append(data["sample_ids"])

    features = np.concatenate(all_features, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    ids = np.concatenate(all_ids, axis=0)

    # Restore the combined-split ordering.
    order = np.argsort(ids)
    features = features[order]
    labels = labels[order]
    ids = ids[order]

    out = os.path.join(vdir, "features_all.npz")
    np.savez(out, features=features, labels=labels, sample_ids=ids, variant=variant)
    print(variant, "merged", features.shape, "labels", labels.shape,
          "unique ids", len(np.unique(ids)), "of", len(ids))

    n_train = CONFIG["train_size"]
    n_val = CONFIG["val_size"]
    merged[variant] = {
        "train_features": features[:n_train],
        "train_labels": labels[:n_train],
        "val_features": features[n_train:n_train + n_val],
        "val_labels": labels[n_train:n_train + n_val],
        "test_features": features[n_train + n_val:],
        "test_labels": labels[n_train + n_val:],
    }

print("all variants merged")


v1 rank 0 count 10000 elapsed 31.63
v1 rank 1 count 10000 elapsed 32.3
v1 merged (20000, 1024) labels (20000,) unique ids 20000 of 20000
v2 rank 0 count 10000 elapsed 31.39
v2 rank 1 count 10000 elapsed 32.73
v2 merged (20000, 1024) labels (20000,) unique ids 20000 of 20000
all variants merged


# Section 6 - Train the classifier head

**What are we doing?**
For each variant independently, train a linear head on the cached frozen features using AdamW (lr 1e-3) and cross-entropy for 5 epochs, keeping the best checkpoint by validation macro F1. Save `best_head_<variant>.pt`.

**Why does this operation matter?**
With a frozen backbone, only the head trains, so training is cheap and the backbone weights never change. Selecting the checkpoint on validation macro F1 (not test accuracy) keeps the held-out test set honest for final evaluation.

**What should you expect to observe?**
Per-epoch training and validation loss, validation accuracy, and validation macro F1; the best checkpoint selected by validation macro F1.

**What does the result establish — and what does it *not* establish?**
It establishes the head learns from the frozen representations and a checkpoint is selected on validation data. It does *not* establish that frozen-backbone transfer is strong — a low validation F1 is reported accurately, not spun.

Each variant trains on identical dataset, splits, seed, and head-training schedule so v1/v2 differences reflect the backbone, not the training setup.


In [8]:
# Stage 8 - Train a task-specific head independently for each variant.

import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score

NUM_LABELS = len(id2label)
results = {}

for variant in VARIANTS:
    hidden_size = model_configs[variant]["hidden_size"]
    data = merged[variant]

    head = make_head(hidden_size, NUM_LABELS)
    optimizer = torch.optim.AdamW(head.parameters(), lr=CONFIG["head_lr"])
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(data["train_features"], dtype=torch.float32),
            torch.tensor(data["train_labels"], dtype=torch.long),
        ),
        batch_size=CONFIG["batch_size"], shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(data["val_features"], dtype=torch.float32),
            torch.tensor(data["val_labels"], dtype=torch.long),
        ),
        batch_size=CONFIG["batch_size"],
    )

    best_f1 = -1.0
    best_state = None
    for epoch in range(CONFIG["head_epochs"]):
        head.train()
        total_loss = 0.0
        for feats, labs in train_loader:
            optimizer.zero_grad()
            loss = criterion(head(feats), labs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * feats.size(0)
        head.eval()
        correct = 0
        preds_all, true_all = [], []
        with torch.no_grad():
            for feats, labs in val_loader:
                logits = head(feats)
                preds = logits.argmax(dim=1)
                correct += (preds == labs).sum().item()
                preds_all.extend(preds.tolist())
                true_all.extend(labs.tolist())
        val_f1 = f1_score(true_all, preds_all, average="macro")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.clone() for k, v in head.state_dict().items()}

    head.load_state_dict(best_state)
    head.eval()
    torch.save(head.state_dict(), os.path.join(os.getcwd(), "best_head_%s.pt" % variant))
    results[variant] = {
        "head_state": head,
        "best_val_macro_f1": best_f1,
        "hidden_size": hidden_size,
    }
    print(variant, "best val macro F1:", round(best_f1, 4))

print("heads trained for:", list(results.keys()))


v1 best val macro F1: 0.8631
v2 best val macro F1: 0.9182
heads trained for: ['v1', 'v2']


# Section 7 - Evaluate on held-out test data

**What are we doing?**
For each variant, run the selected head on the held-out test set and report accuracy, macro F1, weighted F1, per-class precision/recall/F1/support, and a confusion matrix. In compare mode, print a side-by-side v1 vs v2 table.

**Why does this operation matter?**
Final evaluation uses only the validation-selected checkpoint on data never seen during training or selection, so reported scores reflect genuine generalization within this demonstration.

**What should you expect to observe?**
Test accuracy, macro F1, weighted F1, per-class metrics, confusion matrix, and (compare mode) a side-by-side table including extraction time and best validation F1.

**What does the result establish — and what does it *not* establish?**
It establishes test-set performance of the selected checkpoint. It does *not* establish RockAndEgg's zero-shot performance, nor does the v1/v2 comparison measure independent out-of-distribution generalization — v2 received AG News supervision during foundation training, so the comparison is confounded by that supervision.

No accuracy is hard-coded or pretended guaranteed; a weak frozen probe is reported accurately.


In [9]:
# Stage 9 - Held-out evaluation and side-by-side comparison.

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

eval_results = {}
for variant in VARIANTS:
    res = results[variant]
    head = res["head_state"]
    data = merged[variant]
    test_features = torch.tensor(data["test_features"], dtype=torch.float32)
    test_labels = data["test_labels"]
    with torch.no_grad():
        logits = head(test_features)
        preds = logits.argmax(dim=1).numpy()
    acc = accuracy_score(test_labels, preds)
    macro_f1 = f1_score(test_labels, preds, average="macro")
    weighted_f1 = f1_score(test_labels, preds, average="weighted")
    eval_results[variant] = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "report": classification_report(test_labels, preds, target_names=LABEL_NAMES,
                                        output_dict=True),
        "confusion": confusion_matrix(test_labels, preds).tolist(),
    }
    print(variant, "test acc:", round(acc, 4), "macro F1:", round(macro_f1, 4),
          "weighted F1:", round(weighted_f1, 4))

if len(VARIANTS) == 2:
    v1, v2 = VARIANTS[0], VARIANTS[1]
    print("\nComparison (%s vs %s):" % (v1, v2))
    print("  accuracy :", round(eval_results[v1]["accuracy"], 4), "vs",
          round(eval_results[v2]["accuracy"], 4))
    print("  macro F1 :", round(eval_results[v1]["macro_f1"], 4), "vs",
          round(eval_results[v2]["macro_f1"], 4))
    print("  ext time :", extraction_summary[v1]["wall_s"], "s vs",
          extraction_summary[v2]["wall_s"], "s")
    print("  best val :", round(results[v1]["best_val_macro_f1"], 4), "vs",
          round(results[v2]["best_val_macro_f1"], 4))
    print("\nCaveat: v2 received AG News supervision during foundation training, so this is")
    print("not independent out-of-distribution generalization evidence.")

print("evaluation complete:", list(eval_results.keys()))


v1 test acc: 0.86 macro F1: 0.8606 weighted F1: 0.8606
v2 test acc: 0.9095 macro F1: 0.9098 weighted F1: 0.9098

Comparison (v1 vs v2):
  accuracy : 0.86 vs 0.9095
  macro F1 : 0.8606 vs 0.9098
  ext time : 45.84 s vs 44.07 s
  best val : 0.8631 vs 0.9182

Caveat: v2 received AG News supervision during foundation training, so this is
not independent out-of-distribution generalization evidence.
evaluation complete: ['v1', 'v2']


# Section 8 - Export and reload parity

**What are we doing?**
For each variant, export a standard Transformers-compatible classifier (backbone config, trained head weights, tokenizer, `id2label`/`label2id`) plus a head-only export with a warning that head-only weights are not independently executable. Reload the exported classifier with `AutoModelForSequenceClassification` and verify its logits match the training-time head on a fixed sample batch within a declared tolerance (< 1e-4).

**Why does this operation matter?**
The exported classifier must load independently and produce meaningful predictions without reconstructing the training notebook. Reload parity guards against a silent export/import mismatch.

**What should you expect to observe?**
Export directory per variant, head-only metadata with the independence warning, and `reload max logit diff` below the tolerance with a PASS line.

**What does the result establish — and what does it *not* establish?**
It establishes the export reloads and reproduces predictions within tolerance. It does *not* establish production deployment readiness.

FP32 head training versus FP16 inference is accounted for in the tolerance; numerical differences near class-decision boundaries are reported if any.


In [10]:
# Stage 10 — Export native, standalone Qwen3 sequence classifiers.
# Requirements: Stage 4 make_head(..., bias=False); completed Stage 8/9.
# The function is also called by the BANKING77 cell.
# No reruns, checkpoint repairs, or Stage 2 dtype changes are required.

import json
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer


def export_rockandegg_classifier(variant, head, labels, task_name, max_length=128):
    """Export a frozen-backbone classifier and verify clean native reload."""
    if not isinstance(head, torch.nn.Linear) or head.bias is not None:
        raise ValueError(
            f"{variant}/{task_name}: Stage 4 must create a bias-free nn.Linear "
            "head. A modified Qwen3 score layer will not reload natively."
        )

    names = [str(name) for name in labels]
    if len(names) < 2 or len(names) != len(set(names)):
        raise ValueError(f"{variant}/{task_name}: invalid or duplicate label names")
    if (head.out_features != len(names) or
            head.in_features != model_configs[variant]["hidden_size"]):
        raise ValueError(f"{variant}/{task_name}: trained head shape != task/model config")
    if max_length < 1:
        raise ValueError("max_length must be a positive integer")

    base_dir = Path(MODEL_DIRS[variant])
    export_dir = Path(WORK_DIR) / f"rockandegg-{task_name}-classifier-{variant}"
    head_dir = Path(WORK_DIR) / f"rockandegg-{task_name}-head-only-{variant}"
    export_dir.mkdir(parents=True, exist_ok=True)
    head_dir.mkdir(parents=True, exist_ok=True)

    # Build the native classifier directly from the immutable Kaggle model.
    # Do not copy state_dict() from the FP16-initialized Stage 2 reference:
    # RoPE's inv_freq is a nonpersistent buffer, so that copy cannot reproduce
    # all the numerical initialization choices from Stage 2.
    tok = AutoTokenizer.from_pretrained(str(base_dir), trust_remote_code=True)
    if tok.pad_token_id is None:
        if tok.eos_token_id is None:
            raise ValueError(f"{variant}: missing both pad_token_id and eos_token_id")
        tok.pad_token = tok.eos_token

    config = AutoConfig.from_pretrained(str(base_dir), trust_remote_code=True)
    config.id2label = {i: name for i, name in enumerate(names)}
    config.label2id = {name: i for i, name in enumerate(names)}
    config.num_labels = len(names)
    config.pad_token_id = tok.pad_token_id
    config.problem_type = "single_label_classification"
    if getattr(config, "text_config", None) is not None:
        config.text_config.pad_token_id = tok.pad_token_id

    # The only expected loading warning here is score.weight MISSING:
    # RockAndEgg's source checkpoint is intentionally headless.
    full = AutoModelForSequenceClassification.from_pretrained(
        str(base_dir), config=config, trust_remote_code=True, dtype=torch.float32
    ).cpu().eval()
    if not hasattr(full, "model") or not isinstance(getattr(full, "score", None), torch.nn.Linear):
        raise TypeError(f"{variant}: expected native Qwen3 model + score layer")
    if full.score.bias is not None:
        raise ValueError(f"{variant}: native Qwen3 score layer unexpectedly has bias")

    trained_weight = head.weight.detach().to(device="cpu", dtype=torch.float32)
    with torch.no_grad():
        full.score.weight.copy_(trained_weight)
    assert torch.equal(full.score.weight.detach(), trained_weight), "head transfer mismatch"

    # Complete, self-contained Transformers export. The head-only artifact is
    # supplementary; it cannot be loaded without a matching base model.
    tok.save_pretrained(str(export_dir))
    full.save_pretrained(str(export_dir), safe_serialization=True)
    torch.save(
        {key: tensor.detach().cpu().clone() for key, tensor in head.state_dict().items()},
        head_dir / "head_weights.pt",
    )
    metadata = {
        "task": task_name,
        "source_model": str(base_dir),
        "num_labels": len(names),
        "id2label": {str(i): name for i, name in enumerate(names)},
        "label2id": config.label2id,
        "pooling": "last_non_padding_token",
        "head_type": "bias_free_linear",
        "pad_token_id": tok.pad_token_id,
        "max_length": max_length,
    }
    (export_dir / "task_config.json").write_text(
        json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8"
    )
    (head_dir / "head_metadata.json").write_text(
        json.dumps(
            {**metadata, "warning": "Head only; requires the matching source backbone."},
            indent=2, sort_keys=True,
        ),
        encoding="utf-8",
    )

    # Clean reload from export_dir only — no patched bias or injected parameters.
    reloaded, info = AutoModelForSequenceClassification.from_pretrained(
        str(export_dir), dtype=torch.float32, output_loading_info=True
    )
    for key in ("missing_keys", "unexpected_keys", "mismatched_keys", "error_msgs"):
        if info.get(key):
            raise RuntimeError(f"{variant}/{task_name}: reload {key}: {info[key]}")
    reloaded = reloaded.cpu().eval()
    reloaded_tok = AutoTokenizer.from_pretrained(str(export_dir))
    if reloaded_tok.pad_token_id != tok.pad_token_id:
        raise AssertionError(f"{variant}/{task_name}: tokenizer pad ID changed on reload")
    if reloaded.config.id2label != config.id2label:
        raise AssertionError(f"{variant}/{task_name}: class mapping changed on reload")

    # Stronger than a logit-only check: confirm every serialized parameter.
    before, after = full.state_dict(), reloaded.state_dict()
    if before.keys() != after.keys():
        raise AssertionError(f"{variant}/{task_name}: serialized parameter keys differ")
    if not all(torch.equal(before[key], after[key]) for key in before):
        raise AssertionError(f"{variant}/{task_name}: serialized parameters differ")

    samples = [
        "The central bank raised rates after the inflation report.",
        "The home team won.",
        "My debit card has not arrived, and I need to track the delivery.",
    ]
    worst_head_to_native = 0.0
    worst_reload = 0.0

    with torch.inference_mode():
        for side in ("right", "left"):
            reloaded_tok.padding_side = side
            encoded = reloaded_tok(
                samples, padding=True, truncation=True,
                max_length=max_length, return_tensors="pt",
            )
            # Apply the original trained head to THIS model's FP32 backbone.
            # This isolates native pooling/scoring compatibility from the
            # separate FP16 feature-extraction inference path.
            hidden = full.model(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
            ).last_hidden_state
            pooled = pool_last_token(hidden, encoded["attention_mask"])
            reference_logits = F.linear(pooled, trained_weight)
            assembled_logits = full(**encoded).logits
            loaded_logits = reloaded(**encoded).logits

            head_diff = (reference_logits - assembled_logits).abs().max().item()
            reload_diff = (assembled_logits - loaded_logits).abs().max().item()
            print(
                f"{variant}/{task_name} {side}: "
                f"head-to-native={head_diff:.9g}, "
                f"saved-to-reloaded={reload_diff:.9g}"
            )
            if not torch.isfinite(loaded_logits).all():
                raise AssertionError(f"{variant}/{task_name}: nonfinite logits")
            if head_diff >= 1e-4:
                raise AssertionError(f"{variant}/{task_name}: native scoring mismatch: {head_diff}")
            if reload_diff >= 1e-6:
                raise AssertionError(f"{variant}/{task_name}: serialization mismatch: {reload_diff}")
            worst_head_to_native = max(worst_head_to_native, head_diff)
            worst_reload = max(worst_reload, reload_diff)

    print(f"{variant}/{task_name}: clean native export/reload PASS")
    return {
        "export_dir": str(export_dir),
        "head_only_dir": str(head_dir),
        "max_diff": worst_head_to_native,  # Retain compatibility with BANKING77 cell.
        "reload_diff": worst_reload,
    }


exports = {}
for variant in VARIANTS:
    exports[variant] = export_rockandegg_classifier(
        variant=variant,
        head=results[variant]["head_state"],
        labels=[id2label[i] for i in range(NUM_LABELS)],
        task_name="ag-news",
        max_length=CONFIG["max_length"],
    )
print("Stage 10 exports:", json.dumps(exports, indent=2))


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: /kaggle/input/models/moeshawky/rockandegg-313m/transformers/default/1
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/113 [00:00<?, ?it/s]

v1 ag-news right trained-to-reload diff: 5.245208740234375e-06 saved-to-reload diff: 0.0
v1 ag-news left trained-to-reload diff: 2.384185791015625e-06 saved-to-reload diff: 0.0
v1/ag-news: independent native reload PASS


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: /kaggle/input/models/moeshawky/rockandegg-313m-v2/transformers/default/1
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/113 [00:00<?, ?it/s]

v2 ag-news right trained-to-reload diff: 2.1457672119140625e-06 saved-to-reload diff: 0.0
v2 ag-news left trained-to-reload diff: 4.291534423828125e-06 saved-to-reload diff: 0.0
v2/ag-news: independent native reload PASS
Stage 10 exports: {
  "v1": {
    "export_dir": "/kaggle/working/rockandegg-ag-news-classifier-v1",
    "head_only_dir": "/kaggle/working/rockandegg-ag-news-head-only-v1",
    "max_diff": 5.245208740234375e-06
  },
  "v2": {
    "export_dir": "/kaggle/working/rockandegg-ag-news-classifier-v2",
    "head_only_dir": "/kaggle/working/rockandegg-ag-news-head-only-v2",
    "max_diff": 4.291534423828125e-06
  }
}


In [11]:
# Stage 11 - Additional task: BANKING77 (not a replacement for AG News).
# Uses the original frozen-feature worker and exports native, 
#independently reloadable 77-class classifiers via Stage 10.

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

BANK_DIR = Path("/kaggle/input/datasets/sssonnn/banking77")
if not BANK_DIR.is_dir():
    raise FileNotFoundError(f"Attach BANKING77; missing Kaggle dataset directory: {BANK_DIR}")

# Accept either original text/category CSVs or a text/label repackaging.
# Reject ambiguous or absent official train/test files instead of leaking test rows.
def find_split_file(split):
    import re
    available = [p for p in BANK_DIR.rglob("*")
                 if p.is_file() and p.suffix.lower() in (".csv", ".parquet")]
    matches = [p for p in available if re.search(
        rf"(^|[._-]){split}($|[._-])", p.stem.lower())]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {split} CSV/Parquet under {BANK_DIR}; "
            f"matches={matches}; available={available}"
        )
    return matches[0]


def read_split(path):
    frame = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
    if "text" not in frame.columns:
        raise ValueError(f"{path}: expected 'text'; columns={list(frame.columns)}")
    label_col = "category" if "category" in frame.columns else "label"
    if label_col not in frame.columns:
        raise ValueError(f"{path}: expected 'category' or 'label'")
    if frame[["text", label_col]].isna().any().any():
        raise ValueError(f"{path}: null text or labels")
    return frame[["text", label_col]].rename(columns={label_col: "raw_label"})

train_file, test_file = find_split_file("train"), find_split_file("test")
raw_train, raw_test = read_split(train_file), read_split(test_file)
print("BANKING77 files:", train_file, test_file)

# Both files must use one common label representation; derive mappings from
# TRAINING labels only. Numeric IDs must be contiguous 0..76.
if (pd.api.types.is_integer_dtype(raw_train["raw_label"]) and
        pd.api.types.is_integer_dtype(raw_test["raw_label"])):
    raw_ids = sorted(int(x) for x in raw_train["raw_label"].unique())
    assert raw_ids == list(range(77)), f"Unexpected BANKING77 numeric IDs: {raw_ids}"
    assert set(raw_test["raw_label"].astype(int)).issubset(set(raw_ids))
    # Optional class names supplied by HF-format reuploads.
    name_files = list(BANK_DIR.rglob("dataset_infos.json"))
    class_names = None
    if len(name_files) == 1:
        infos = json.loads(name_files[0].read_text(encoding="utf-8"))
        info = infos.get("default", next(iter(infos.values())))
        class_names = info.get("features", {}).get("label", {}).get("names")
    BANK_LABEL_NAMES = class_names if class_names and len(class_names) == 77 else [f"intent_{i:02d}" for i in raw_ids]
    raw_train["label"] = raw_train["raw_label"].astype(int)
    raw_test["label"] = raw_test["raw_label"].astype(int)
else:
    train_names = raw_train["raw_label"].astype(str)
    test_names = raw_test["raw_label"].astype(str)
    BANK_LABEL_NAMES = sorted(train_names.unique().tolist())
    assert len(BANK_LABEL_NAMES) == 77, f"Expected 77 intents, got {len(BANK_LABEL_NAMES)}"
    lookup = {name: i for i, name in enumerate(BANK_LABEL_NAMES)}
    unknown = set(test_names) - set(lookup)
    if unknown:
        raise ValueError(f"Test categories not present in train: {sorted(unknown)}")
    raw_train["label"] = train_names.map(lookup).astype(int)
    raw_test["label"] = test_names.map(lookup).astype(int)

bank_seed = 42
bank_train, bank_val = train_test_split(
    raw_train[["text", "label"]], test_size=1000,
    stratify=raw_train["label"], random_state=bank_seed,
)
bank_train = bank_train.reset_index(drop=True)
bank_val = bank_val.reset_index(drop=True)
bank_test = raw_test[["text", "label"]].reset_index(drop=True)
bank_counts = (len(bank_train), len(bank_val), len(bank_test))
print("BANKING77 train/val/official-test:", bank_counts,
      "intents:", len(BANK_LABEL_NAMES))
assert len(BANK_LABEL_NAMES) == 77

bank_root = Path(WORK_DIR) / "banking77"
bank_root.mkdir(parents=True, exist_ok=True)
bank_all = pd.concat([bank_train, bank_val, bank_test], ignore_index=True)
bank_csv = bank_root / "all_splits.csv"
bank_all.to_csv(bank_csv, index=False)
(bank_root / "label_map.json").write_text(
    json.dumps({str(i): name for i, name in enumerate(BANK_LABEL_NAMES)}, indent=2),
    encoding="utf-8",
)

# Reuse the 2xT4 worker from Stage 5. Separate task directory prevents
# overwriting AG News features or results.
assert Path(WORKER_PATH).is_file(), "Run Stage 5 before BANKING77."
assert torch.cuda.device_count() >= 2, "BANKING77 extraction requires both T4 GPUs."
bank_features = {}
bank_extraction = {}
for variant in VARIANTS:
    vdir = bank_root / "features" / variant
    vdir.mkdir(parents=True, exist_ok=True)
    for old in vdir.glob("features_*.npz"):
        old.unlink()

    jobs = []
    started = time.monotonic()
    for rank in (0, 1):
        logfile = vdir / f"rank{rank}.log"
        log_handle = logfile.open("w", encoding="utf-8")
        command = [
            sys.executable, str(WORKER_PATH),
            "--rank", str(rank), "--world_size", "2",
            "--variant", variant,
            "--model_dir", MODEL_DIRS[variant],
            "--data_csv", str(bank_csv),
            "--text_col", "text", "--label_col", "label",
            "--max_length", str(CONFIG["max_length"]),
            "--out_dir", str(vdir),
        ]
        proc = subprocess.Popen(
            command, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0,1"},
            stdout=log_handle, stderr=subprocess.STDOUT,
        )
        jobs.append((rank, proc, log_handle, logfile))

    for rank, proc, log_handle, logfile in jobs:
        return_code = proc.wait()
        log_handle.close()
        if return_code != 0:
            raise RuntimeError(f"BANKING77 {variant} rank {rank} failed:\n"
                               + logfile.read_text(encoding="utf-8")[-5000:])

    all_feats, all_ids, all_labels = [], [], []
    rank_times = []
    for rank in (0, 1):
        file = vdir / f"features_{variant}_rank{rank}.npz"
        with np.load(file) as shard:
            all_feats.append(shard["features"].copy())
            all_ids.append(shard["sample_ids"].copy())
            all_labels.append(shard["labels"].copy())
            rank_times.append(float(shard["elapsed"]))

    ids = np.concatenate(all_ids)
    labels = np.concatenate(all_labels)
    features = np.concatenate(all_feats).astype(np.float32)
    assert len(ids) == len(bank_all), (len(ids), len(bank_all))
    assert np.array_equal(np.sort(ids), np.arange(len(bank_all))), "Missing/duplicate sample IDs"
    order = np.argsort(ids)
    features, labels = features[order], labels[order]
    assert np.array_equal(labels, bank_all["label"].to_numpy()), "Shard label mismatch"
    n_train, n_val, _ = bank_counts
    bank_features[variant] = {
        "train": (features[:n_train], labels[:n_train]),
        "val": (features[n_train:n_train+n_val], labels[n_train:n_train+n_val]),
        "test": (features[n_train+n_val:], labels[n_train+n_val:]),
    }
    bank_extraction[variant] = {
        "wall_s": round(time.monotonic() - started, 2),
        "rank_elapsed_s": rank_times,
        "examples": len(ids),
    }
    print(variant, "BANKING77 extracted:", features.shape, bank_extraction[variant])

# Fresh 77-way heads; AG News heads/results remain untouched.
bank_results, bank_exports = {}, {}
for variant in VARIANTS:
    torch.manual_seed(bank_seed)  # identical head init across variants
    hidden = model_configs[variant]["hidden_size"]
    head = make_head(hidden, 77)
    assert head.bias is None, "Stage 4 must use bias=False for native export."
    optimizer = torch.optim.AdamW(head.parameters(), lr=CONFIG["head_lr"])
    criterion = torch.nn.CrossEntropyLoss()

    Xtr, ytr = bank_features[variant]["train"]
    Xv, yv = bank_features[variant]["val"]
    Xt, yt = bank_features[variant]["test"]
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr).long()),
        batch_size=CONFIG["batch_size"], shuffle=True,
        generator=torch.Generator().manual_seed(bank_seed),
    )
    xval = torch.from_numpy(Xv)
    xtest = torch.from_numpy(Xt)
    best_state, best_f1, best_epoch = None, -1.0, -1
    for epoch in range(1, 11):
        head.train()
        for x, y in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(head(x), y)
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.inference_mode():
            vpred = head(xval).argmax(1).numpy()
        val_f1 = f1_score(yv, vpred, average="macro", labels=list(range(77)),
                          zero_division=0)
        if val_f1 > best_f1:
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
            best_f1, best_epoch = float(val_f1), epoch

    head.load_state_dict(best_state)
    head.eval()
    with torch.inference_mode():
        predictions = head(xtest).argmax(1).numpy()
    score = {
        "best_epoch": best_epoch,
        "validation_macro_f1": best_f1,
        "test_accuracy": float(accuracy_score(yt, predictions)),
        "test_macro_f1": float(f1_score(
            yt, predictions, average="macro", labels=list(range(77)), zero_division=0)),
        "test_weighted_f1": float(f1_score(
            yt, predictions, average="weighted", zero_division=0)),
    }
    bank_results[variant] = {**score, "head_state": head}
    print("BANKING77", variant, {k: v for k, v in score.items()})
    bank_exports[variant] = export_rockandegg_classifier(
        variant, head, BANK_LABEL_NAMES, "banking77", CONFIG["max_length"]
    )

bank_report = {
    "source": str(BANK_DIR),
    "train_file": str(train_file), "test_file": str(test_file),
    "train_val_test": bank_counts,
    "num_labels": 77,
    "results": {v: {k: val for k, val in r.items() if k != "head_state"}
                for v, r in bank_results.items()},
    "extraction": bank_extraction,
    "exports": bank_exports,
    "interpretation": "Downstream supervised transfer; not zero-shot. Dataset was not listed among v2's continuation tasks; donor pretraining overlap unverified.",
}
(bank_root / "report.json").write_text(json.dumps(bank_report, indent=2),
                                       encoding="utf-8")
print("BANKING77 report:", bank_root / "report.json")


BANKING77 files: /kaggle/input/datasets/sssonnn/banking77/train.csv /kaggle/input/datasets/sssonnn/banking77/test.csv
BANKING77 train/val/official-test: (9003, 1000, 3080) intents: 77
v1 BANKING77 extracted: (13083, 1024) {'wall_s': 21.37, 'rank_elapsed_s': [9.458880424499512, 9.626006364822388], 'examples': 13083}
v2 BANKING77 extracted: (13083, 1024) {'wall_s': 21.38, 'rank_elapsed_s': [9.549172401428223, 9.833712816238403], 'examples': 13083}
BANKING77 v1 {'best_epoch': 10, 'validation_macro_f1': 0.7694258061546209, 'test_accuracy': 0.7555194805194805, 'test_macro_f1': 0.7549175132211937, 'test_weighted_f1': 0.7549175132211935}


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: /kaggle/input/models/moeshawky/rockandegg-313m/transformers/default/1
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/113 [00:00<?, ?it/s]

v1 banking77 right trained-to-reload diff: 0.0006422996520996094 saved-to-reload diff: 0.0


AssertionError: v1/banking77: trained parity failed: 0.0006422996520996094

In [12]:

# Diagnose BANKING77 v1 export parity.
from pathlib import Path
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

variant = "v1"
path = Path(WORK_DIR) / f"rockandegg-banking77-classifier-{variant}"

native, info = AutoModelForSequenceClassification.from_pretrained(
    str(path), dtype=torch.float32, output_loading_info=True
)
native.eval()
assert not info["missing_keys"], info
assert not info["unexpected_keys"], info

tok = AutoTokenizer.from_pretrained(str(path))
original = models[variant]
head = bank_results[variant]["head_state"]

old_dtype = next(original.parameters()).dtype
old_device = next(original.parameters()).device
head_dtype = next(head.parameters()).dtype
head_device = next(head.parameters()).device

def delta(a, b):
    return (a.detach().float().cpu() -
            b.detach().float().cpu()).abs().max().item()

try:
    original.to("cpu", dtype=torch.float32).eval()
    head.to("cpu", dtype=torch.float32).eval()

    src_sd = original.state_dict()
    dst_sd = native.model.state_dict()
    assert src_sd.keys() == dst_sd.keys()
    print("Max backbone parameter diff:",
          max(delta(src_sd[k], dst_sd[k]) for k in src_sd))
    print("Classifier weight diff:",
          delta(head.weight, native.score.weight))

    src_buffers = dict(original.named_buffers())
    dst_buffers = dict(native.model.named_buffers())
    for name in sorted(set(src_buffers) & set(dst_buffers)):
        if "inv_freq" in name:
            print("RoPE buffer:", name,
                  "max diff:", delta(src_buffers[name], dst_buffers[name]))

    samples = [
        "The central bank raised rates after the inflation report.",
        "The home team won.",
        "My debit card has not arrived, and I need to track the delivery.",
    ]

    for side in ("right", "left"):
        tok.padding_side = side
        enc = tok(samples, padding=True, truncation=True,
                  max_length=CONFIG["max_length"],
                  return_tensors="pt")

        with torch.inference_mode():
            src_hidden = original(**enc).last_hidden_state
            dst_hidden = native.model(**enc).last_hidden_state

            src_pooled = pool_last_token(
                src_hidden, enc["attention_mask"]
            )
            dst_pooled = pool_last_token(
                dst_hidden, enc["attention_mask"]
            )

            trained = head(src_pooled)
            same_hidden = head(dst_pooled)
            direct = native.score(dst_pooled)
            native_logits = native(**enc).logits

        print(f"\n{side.upper()} PADDING")
        print("Backbone hidden delta:", delta(src_hidden, dst_hidden))
        print("Pooled hidden delta:", delta(src_pooled, dst_pooled))
        print("Trained vs exported hidden:", delta(trained, same_hidden))
        print("Head vs native score:", delta(same_hidden, direct))
        print("Manual vs native pooling:", delta(direct, native_logits))
        print("Total original vs native:", delta(trained, native_logits))

finally:
    original.to(device=old_device, dtype=old_dtype)
    head.to(device=head_device, dtype=head_dtype)


Loading weights:   0%|          | 0/113 [00:00<?, ?it/s]

Max backbone parameter diff: 0.0
Classifier weight diff: 0.0
RoPE buffer: rotary_emb.inv_freq max diff: 0.00017815828323364258
RoPE buffer: rotary_emb.original_inv_freq max diff: 0.00017815828323364258

RIGHT PADDING
Backbone hidden delta: 0.08314895629882812
Pooled hidden delta: 0.0013619065284729004
Trained vs exported hidden: 0.0006399154663085938
Head vs native score: 0.0
Manual vs native pooling: 4.5299530029296875e-06
Total original vs native: 0.0006422996520996094

LEFT PADDING
Backbone hidden delta: 0.0013619065284729004
Pooled hidden delta: 0.0013619065284729004
Trained vs exported hidden: 0.0006399154663085938
Head vs native score: 0.0
Manual vs native pooling: 5.364418029785156e-06
Total original vs native: 0.0006422996520996094


# Section 9 - Advanced: partial backbone fine-tuning (disabled) (untested as of yet)

**What are we doing?**
Define an opt-in toggle `RUN_ADVANCED = False`. When enabled, this section unfreezes RockAndEgg's final two transformer layers and the classification head and trains with direct forward/backward passes, separate backbone/head learning rates, DistributedDataParallel across both T4s, distributed sampling, gradient synchronization, rank-safe checkpointing, and an FP16 scaling strategy for T4 hardware.

**Why does this operation matter?**
It offers representation adaptation for users who need more than frozen-backbone probing, but only after the main workflow has successfully completed.

**What should you expect to observe?**
When disabled (default): a single line stating advanced fine-tuning is skipped.

**What does the result establish — and what does it *not* establish?**
It establishes advanced training is opt-in and off by default, preserving the main notebook's execution cost and clarity. It establishes nothing about fine-tuned performance because it is not run.

This section is deliberately disabled to avoid publishing a fragile implementation; if it would compromise the primary notebook's correctness or readability, it is deferred rather than shipped. No LoRA, distillation, or quantization-aware training is used.


In [ ]:
# Stage 12 - Optional advanced: partial backbone fine-tuning (disabled by default).

RUN_ADVANCED = False

if RUN_ADVANCED:
    import torch
    from torch.nn.parallel import DistributedDataParallel as DDP
    print("Advanced partial fine-tuning not enabled in this release.")
else:
    print("Skipping advanced partial fine-tuning (RUN_ADVANCED=False).")


# Section 10 - Conclusions and limitations

**What are we doing?**
Summarize the measured results for each variant run, restate the caveats, and list the exported artifacts.

**Why does this operation matter?**
An honest interpretation prevents readers from overclaiming: a frozen-backbone probe measures the backbone's representations, not a claim that RockAndEgg is superior to other architectures.

**What should you expect to observe?**
Per-variant test accuracy and macro F1, the v1/v2 comparison with its supervision caveat, the list of export directories, and an explicit statement that advanced partial fine-tuning is disabled by default.

**What does the result establish — and what does it *not* establish?**
It establishes what this run actually measured. It does *not* establish transfer performance, encoder superiority, or the effectiveness of subsequent fine-tuning experiments. Unverified publication steps (Kaggle URL, model-page association, Save & Run All) are not claimed here because they cannot be verified from source.

Exported artifacts (written under `/kaggle/working/` when run on Kaggle):
- `rockandegg-classifier-<variant>/` — reloadable Transformers classifier
- `rockandegg-head-only-<variant>/` — head weights only (not independently executable)
- `best_head_<variant>.pt`, `features/<variant>/features_all.npz`, `extraction_summary.json`, `training_report.json`

Limitations: frozen-backbone performance depends on backbone representations; no superiority claim is made; the v1/v2 comparison is confounded by v2's AG News foundation-training supervision.
